# CondOT Flow Matching ODE 求解器：自包含学习路线

这些 notebook 不依赖仓库中的 Python 模块；每一份都只导入 PyTorch，并重复定义自身所需的 solver、时间网格和测试。可以单独复制任意一个到新目录运行。

共同的数学起点是：CondOT 路径 `x_t=(1-t)x_noise+t x_data`，训练得到的 decoder 给出速度场 `vθ(x,t)`，推理阶段求解 `dx/dt=vθ(x,t)`，从 t=0 前进到 t=1。encoder 一次的上下文可以缓存；每一次速度函数计算才会调用 decoder。

## 如何公平比较

NFE 是速度场函数评价次数，也就是大模型 decoder forward 次数。固定 steps 并不公平：Euler 的 10 steps 为 10 NFE，Heun/Midpoint 的 5 steps 才是 10 NFE，RK4 的 2 steps 是 8 NFE。比较质量和时延时，应首先固定 NFE，再报告实际毫秒数。

| 方法 | 全局阶 | 新 NFE/步 | 核心思想 |
| --- | --- | --- | --- |
| Euler | 1 | 1 | 左端速度冻结 |
| Midpoint | 2 | 2 | 半步预测，中点速度 |
| Heun | 2 | 2 | Euler 预测，再作端点平均 |
| RK4 | 4 | 4 | 四个 stage 的加权平均 |
| AB2 | 2 | 1 | 缓存前一步 raw velocity |
| CondOT-2M | 2 | 1 | 缓存 data prediction 并解析积分线性项 |

推荐阅读顺序为 01 Euler、02 Midpoint、03 Heun、04 RK4、05 AB2、06 CondOT-2M。每份最后均有恒速度不变量测试和一个 `v=x` 或端点稳定性测试。


## 与当前工程的对应关系和边界

本系列 `01` 至 `06` 的主体是 CondOT Flow Matching：它们从同一个初值问题

$$
\frac{d x_t}{dt}=v_\theta(x_t,t,c),\qquad x_0\sim\mathcal N(0,I),
$$

出发，但只改变积分器。仓库中的对应实现位于 `solvers/`：`euler.py`、`midpoint.py`、`heun.py`、`rk4.py`、`ab2.py` 和 `condot_2m.py`。所有实现都把 encoder 上下文缓存一次；NFE 只统计 decoder 的调用。

`07_vpsde_dpm_solver.ipynb` 讨论 VP-SDE probability-flow ODE、`noise/x_start/score/pf_velocity` 的转换，以及 DPM-Solver 2M 与 DPM-Solver++ 2M 为什么偏好不同表示。

端点是一个必须说明的实现细节。直线 CondOT 路径的 data-to-velocity 转换含有 $1/(1-t)$，score-to-velocity 转换含有 $1/t$。因此网络训练和 decoder 输入限定在有限内部区间；数值积分的物理时间仍可达到端点，但不能把未定义的参数变换当作普通浮点运算处理。
